In [1]:
import os
from typing import TypedDict, Literal, Annotated
from io import StringIO
from pathlib import Path
from pprint import pprint


from dotenv import load_dotenv

import nltk
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title
import pandas as pd

from pydantic import BaseModel
# from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import PromptTemplate

/Users/amitkumarmahapatra/Documents/projects/genai-delta-generator/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
for pkg in ["punkt", "punkt_tab", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.data.find(f"tokenizers/{pkg}")
    except LookupError:
        nltk.download(pkg, quiet=True)

In [4]:
def partition_a_pdf(filepath: str):
    elements = partition_pdf(
        filename=filepath,
        strategy="hi_res",
        infer_table_structure=True,
        extract_image_block_types=["Image"],
        extract_image_block_to_payload=True,
        languages=['English']
    )
    return elements

In [5]:
# Using Ollama
# model = ChatOllama(model='gemma2:9b')
# response =  model.invoke('What is the capital of India')
# response.content

# Using Huggingface
llm = HuggingFaceEndpoint(repo_id='google/gemma-2-9b-it', task='text-generation')
model = ChatHuggingFace(llm=llm)
response = model.invoke('What is the capital of India')
response.content

'The capital of India is **New Delhi**. \n'

In [6]:
global_sop_filepath = Path().cwd() / 'data' / 'global-sop.pdf'
site_document_filepath = Path().cwd() / 'data' / 'hdfc-ergo-health-insurance.pdf'

## Step 1: Get "Global SOP Context Table" and "Site Document Title"

In [7]:
global_sop_elements = partition_a_pdf(global_sop_filepath)
global_sop_elements

The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


In [8]:
# # I know that there is only 1 table in global SOP and that is the SOP Context table
for global_sop_element in global_sop_elements:
    if global_sop_element.category == 'Table':
        table_obj = global_sop_element.to_dict()
        table_html = table_obj['metadata']['text_as_html']
        global_sop_context_df = pd.read_html( StringIO(table_html) )[0]
        break
global_sop_context_df

,Sector,Focus Area,Policy Documents
0,Health Insurance,Coverage Limitations Eligibility Criteria Netw...,HI-101: Individual Health Insurance Policy HI-...
1,Vehicle Insurance,Premium Evaluation Framework Coverage Restrict...,VI-101: Comprehensive Motor Insurance Policy V...
2,Life Insurance,Benefit Payout Scope Applicant Eligibility Sta...,LI-101: Term Life Insurance Agreement LI-201: ...


In [9]:
side_doc_elements = partition_a_pdf(site_document_filepath)
side_doc_elements

In [10]:
# # I know in Site Document, the first table will contain title in row 2, column 2

for side_doc_element in side_doc_elements:
    if side_doc_element.category == 'Table':
        table_obj = side_doc_element.to_dict()
        table_html = table_obj['metadata']['text_as_html']
        site_doc_title_df = pd.read_html( StringIO(table_html) )[0]
        break
site_doc_title_df

,0,1
0,Title,General Individual Health Insurance
1,Company,HDFC Ergo


In [11]:
class RelevantRow(TypedDict):
    relevance: bool

In [12]:
structured_model = model.with_structured_output(schema=RelevantRow)

In [13]:
prompt_template = PromptTemplate(
    template='''
    You will be given a table and a title.
    By looking at the table and title, you need to respond if the table is completly relevant to the title or not.
    Here is the table in html tag
    {table_in_html}
    Here is the title
    {title}
    ''',
    input_variables=['table_in_html', 'title']
)

In [14]:
chain = prompt_template | structured_model

In [15]:
site_doc_title = site_doc_title_df.iloc[ 0, 1 ]
site_doc_title

'General Individual Health Insurance'

In [16]:
row_index_to_drop = []

In [17]:
for idx, row in global_sop_context_df.iterrows():
    table_row_html = row.to_frame().to_html(index=False)
    payload = { 'table_in_html': table_row_html, 'title': site_doc_title }
    response = chain.invoke(payload)
    if response.get('relevance') is False:
        row_index_to_drop.append(idx)

In [18]:
row_index_to_drop

[1, 2]

In [19]:
for idx in row_index_to_drop:
    global_sop_context_df.drop(idx, inplace=True)

In [20]:
global_sop_context_df

,Sector,Focus Area,Policy Documents
0,Health Insurance,Coverage Limitations Eligibility Criteria Netw...,HI-101: Individual Health Insurance Policy HI-...


## Step 2: Update "Global SOP Context Table" - Policy Document Column

In [21]:
class RelevantPolicyDocument(TypedDict):
    relevant_existing_policy_document: str

In [22]:
filter_policy_document_prompt_template = PromptTemplate(
    template='''
    You will be given 2 strings 
    existing_policy_document_titles: It is basically a list of policy documents which already exists.
    new_policy_document_title: Title of the new policy document.
    Now you need to return a subset of existing_policy_document_titles separated by "," 
    which match exactly same as the new_policy_document_title semantically.
    Here is the existing policy document titles
    {existing_policy_document_titles}
    Here is the new policy document title
    {new_policy_document_title}
    Rules: Please provide entire title of the existing document not only the id or only the title.
    Example 'VI-301: Commercial Vehicle Insurance Package'
    ''',
    input_variables=['existing_policy_document_titles', 'new_policy_document_title']
)

In [23]:
structured_model = model.with_structured_output(RelevantPolicyDocument)

In [24]:
relevant_policy_document_chain = filter_policy_document_prompt_template | structured_model

In [25]:
for idx, row in global_sop_context_df.iterrows():
    payload = { 'existing_policy_document_titles': row.iloc[2] , 'new_policy_document_title': site_doc_title }
    response = relevant_policy_document_chain.invoke(payload)
    global_sop_context_df.iloc[ idx, 2 ] = response.get('relevant_existing_policy_document')

In [26]:
global_sop_context_df

,Sector,Focus Area,Policy Documents
0,Health Insurance,Coverage Limitations Eligibility Criteria Netw...,HI-101: Individual Health Insurance Policy


## Step 3: Update "Global SOP Context Table" - Focus Area Column

In [27]:
class RelevantFocusArea(TypedDict):
    relevant_focus_areas: str

In [28]:
filter_focus_area_prompt_template = PromptTemplate(
    template='''
    You will be given 2 strings 
    relevant_policy_document_titles: It is basically a list of policy documents, separated by ","
    list_of_focus_areas: It is a list of focus areas. This list may include some focus areas which are not at all related to any document titles.
    Now you need to return a subset of list_of_focus_areas, MUST be separated by comma(",") 
    which is relevant to atleaset one of the relevant policy documents.
    Here is the list of relevant policy document titles
    {relevant_policy_document_titles}
    Here is the list of focus areas
    {focus_areas}
    ''',
    input_variables=['relevant_policy_document_titles', 'focus_areas']
)

In [29]:
structured_model = model.with_structured_output(RelevantFocusArea)

In [30]:
relevant_focus_area_chain = filter_focus_area_prompt_template | structured_model

In [31]:
for idx, row in global_sop_context_df.iterrows():
    focus_areas, relevant_policy_document_titles = row.iloc[1], row.iloc[2]
    payload = { 'relevant_policy_document_titles': relevant_policy_document_titles , 'focus_areas': focus_areas }
    response = relevant_focus_area_chain.invoke(payload)
    global_sop_context_df.iloc[ idx, 1 ] = response.get('relevant_focus_areas')

In [32]:
global_sop_context_df

,Sector,Focus Area,Policy Documents
0,Health Insurance,"Coverage Limitations, Eligibility Criteria, Ne...",HI-101: Individual Health Insurance Policy


## Step 4: Site Document Content Preparation

In [33]:
site_doc_context = '\n'.join([element.text for element in side_doc_elements[2:]])
site_doc_context

'Claims Process ......................................................................................................... 2\nOverview\n1.1 Provides financial protection for hospitalization and medically necessary treatments as per approved policy conditions.\n1.2 Includes cashless treatment at network hospitals subject to prior authorization.\n1.3 Offers reimbursement-based claims for out-of-network treatments.\nEligibility & Enrollment\n2.1 Adults aged 18–65 years are eligible for enrollment subject to underwriting. 2.2 Dependent children between 90 days and 25 years can be added under a family oater. 2.3 Enrollment requires compliance with KYC norms and acceptance of policy declarations.\nRequired Documentation\n3.1 Previous medical history records including surgeries, chronic conditions, and medication details.\n3.2 Diagnostic reports such as blood tests, ECG, or physician evaluation when required. 3.3 Identity proof, address proof, and prior health insurance policy documents.\nPrem

## Step 5: Replicate Filtered Global SOP Table and Add 'Status' and 'Comments' Column

In [34]:
delta_table = global_sop_context_df.copy()
delta_table["Status"] = ""
delta_table["Comments"] = ""
delta_table

,Sector,Focus Area,Policy Documents,Status,Comments
0,Health Insurance,"Coverage Limitations, Eligibility Criteria, Ne...",HI-101: Individual Health Insurance Policy,,


## Step 6: Create Delta Table

In [35]:
class Delta(TypedDict):
    included: Literal[ 'Yes', 'No', 'Partially' ]
    comment: Annotated[str, "A comment if focus area is included but partially"]

In [36]:
delta_prompt_template = PromptTemplate(
    template='''
    You will be given 2 stings.
    Site Document Content: This is the content is extracted from a document which contains details on several focus areas.
    Focus Area: This is a specific focus area.
    Given the Site Document Content and Focus Area, please evaluate strictly whether the given focus area is properly present or partiall present or not present at all.
    If the focus area is partially present, provide a one-liner comment on what is missing.
    If the focus area is not at all present or completely present, no need to provide any comment.

    Here is the Site Document Content
    {site_document_content}
    
    Focus Area
    {focus_area}
    ''',
    input_variables=['site_document_content', 'focus_area']
)

In [37]:
structured_model = model.with_structured_output(schema=Delta)

In [38]:
delta_generation_chain = delta_prompt_template | structured_model

In [39]:
for idx, row in global_sop_context_df.iterrows():
    focus_areas = row.iloc[1].split(',')
    status, comments = [], []
    for focus_area in focus_areas:
        payload = { 'focus_area': focus_area, 'site_document_content': site_doc_context }
        response = delta_generation_chain.invoke(payload)
        status.append( f'{focus_area}: {response.get("included")}' )
        if response.get('included') == 'Partially':
            comments.append(f'{focus_area}: {response.get("comment")}')
    delta_table.iloc[idx, 3] = "\n".join(status)
    delta_table.iloc[idx, 4] = "\n".join(comments)

In [40]:
delta_table

,Sector,Focus Area,Policy Documents,Status,Comments
0,Health Insurance,"Coverage Limitations, Eligibility Criteria, Ne...",HI-101: Individual Health Insurance Policy,Coverage Limitations: Partially\n Eligibility ...,Coverage Limitations: Missing details on waiti...


In [41]:
# site_doc_chunks = chunk_by_title(side_doc_elements[2:])
# for site_chunk in site_doc_chunks:
#     print(site_chunk.text)
#     print('----')

In [42]:
# # Code to convert pandas df back to html table tags
html_out = delta_table.to_html(index=False)
print(html_out)

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>Sector</th>
      <th>Focus Area</th>
      <th>Policy Documents</th>
      <th>Status</th>
      <th>Comments</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Health Insurance</td>
      <td>Coverage Limitations, Eligibility Criteria, Network, Service Accessibility, Premium Structure, Benefit Coverage Scope, Cost-Sharing Terms, Pre-Existing Condition Rules, Claims Handling Process, Waiting Period Guidelines</td>
      <td>HI-101: Individual Health Insurance Policy</td>
      <td>Coverage Limitations: Partially\n Eligibility Criteria: Partially\n Network: Partially\n Service Accessibility: Partially\n Premium Structure: Partially\n Benefit Coverage Scope: Partially\n Cost-Sharing Terms: Partially\n Pre-Existing Condition Rules: Partially\n Claims Handling Process: Partially\n Waiting Period Guidelines: Partially</td>
      <td>Coverage Limitations: Missing details on waiting periods for s